In [10]:
import os
import joblib
import streamlit as st
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error,root_mean_squared_error
from sklearn.model_selection import cross_val_score
import seaborn as sb


PIPELINE_FILE="pipeline2.pkl"
MODEL_FILE="model2.pkl"
def pipeline_bulit(datas,data_label):
    num_pipeline=Pipeline([
    ("standard",StandardScaler())])
    cal_pipeline=Pipeline([
    ("autoCoder",OneHotEncoder(handle_unknown="ignore"))])
    full_pipeline=ColumnTransformer([
    ("num",num_pipeline,datas),
    ("cat",cal_pipeline,data_label)])
    return full_pipeline 

data = pd.read_csv("air_quality_index.csv")
data["pollution_cons"]= pd.cut(data["pollutant_avg"],bins=[0,10,20,30,40,50,np.inf],labels=[1,2,3,4,5,6])
num_data = data.select_dtypes(include=[np.number])
cat_data=data.select_dtypes(exclude=[np.number])

imputer = SimpleImputer(strategy="mean")
x=imputer.fit_transform(num_data)
datan=pd.DataFrame(x,columns=num_data.columns,index=data.index)
imputer2 = SimpleImputer(strategy="most_frequent")
x2=imputer2.fit_transform(cat_data)
datac=pd.DataFrame(x2,columns=cat_data.columns,index=data.index)
data_clean=pd.concat([datac,datan],axis=1)
    
split=StratifiedShuffleSplit(n_splits=1,test_size=0.2,random_state=42)
for train_index,test_index in split.split(data_clean,data_clean["pollution_cons"]):
    strat_train=data_clean.loc[train_index].drop("pollution_cons",axis=1)
    strat_test=data_clean.loc[test_index].drop("pollution_cons",axis=1)


if not os.path.exists(MODEL_FILE):
    
    datatr=strat_train.copy()
    data_label=datatr["pollutant_avg"].copy()
    datatr=datatr.drop("pollutant_avg",axis=1)
    
    
    num1_data=datatr.select_dtypes(include=[np.number]).columns.tolist()
    cat1_data=datatr.select_dtypes(exclude=[np.number]).columns.tolist()

    
    pipeline21=pipeline_bulit(num1_data,cat1_data)
    pollution_found=pipeline21.fit_transform(datatr)

    models=RandomForestRegressor(random_state=42)
    full_prepared=models.fit(pollution_found,data_label)
    
    Forest=RandomForestRegressor(random_state=42)
    dataset=Forest.fit(pollution_found,data_label)
    predict=dataset.predict(pollution_found)
    random=mean_squared_error(predict,data_label)
    
    scoring_random=cross_val_score(
    Forest,pollution_found,data_label,cv=10,scoring="neg_mean_squared_error")
    print(f"The tree score is : {scoring_random}")
    print(pd.Series(scoring_random).describe())

    joblib.dump(models,MODEL_FILE)
    joblib.dump(pipeline21,PIPELINE_FILE)
    print("Your model is trained and fully functional")
else:
    model1=joblib.load(MODEL_FILE)
    pipeline2=joblib.load(PIPELINE_FILE)
    # input_data=pd.read_csv("air_pollution.csv")
    transform_data=pipeline2.transform(strat_test)
    prediction=model1.predict(transform_data)

    strat_test["pollution_avg"]=prediction
    strat_test.to_csv("output_air_pollution.csv",index=False)
    print("work is done with full efficeincy")
    sns.lineplot(x=datatr["pollution_avg"].min(),y=datatr["polltuion_avg"].max()


    

    
    

work is done with full efficeincy
